# 04 — Model Evaluation

Comprehensive evaluation on the held-out test set:
- Quadratic Weighted Kappa (primary clinical metric)
- Per-class Precision / Recall / F1
- Confusion matrices (absolute + normalised)
- Per-class ROC curves with AUC
- Misclassified image gallery

In [ ]:
import os, sys, importlib, shutil

# --- CONFIGURE DATA SOURCE ---
# "kaggle" = download from Kaggle (~1-3 min, recommended for Colab)
# "drive"  = copy from Google Drive (~15-30 min, fallback)
# "local"  = data already at expected path (no copy needed)
DATA_SOURCE = "kaggle"

IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/diabetic_retinopathy'
    %pip install -q timm albumentations

    LOCAL_DATA = '/content/dataset'
    if not os.path.isdir(os.path.join(LOCAL_DATA, 'resized_train_cropped')):
        if DATA_SOURCE == "kaggle":
            print('Upload your kaggle.json when prompted...')
            from google.colab import files
            uploaded = files.upload()  # user uploads kaggle.json
            os.makedirs('/root/.kaggle', exist_ok=True)
            shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            !kaggle datasets download -d tanlikesmath/diabetic-retinopathy-resized -p /content/ -q
            os.makedirs(LOCAL_DATA, exist_ok=True)
            import zipfile
            with zipfile.ZipFile('/content/diabetic-retinopathy-resized.zip', 'r') as z:
                z.extractall(LOCAL_DATA)
            os.remove('/content/diabetic-retinopathy-resized.zip')
            print('Kaggle download complete!')
        elif DATA_SOURCE == "drive":
            print('Copying dataset from Drive to local SSD (15-30 min)...')
            shutil.copytree(
                os.path.join(PROJECT_ROOT, 'dataset'), LOCAL_DATA,
                dirs_exist_ok=True,
            )
            print('Drive copy complete!')
    else:
        print('Dataset already on local SSD.')
else:
    PROJECT_ROOT = os.path.abspath('..')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config

print(f'Images dir: {config.IMAGES_DIR}')
print(f'  exists: {os.path.isdir(config.IMAGES_DIR)}')
print(f'Outputs dir: {config.OUTPUTS_DIR}')

In [ ]:
import torch
import pandas as pd

from src import config
from src.utils import seed_everything, get_device
from src.dataset import DRDataset, create_dataloaders
from src.augmentations import get_valid_transforms
from src.model import build_model
from src.evaluate import full_evaluation, plot_training_curves

seed_everything()
device = get_device()

## 1. Load Test Data and Model

In [ ]:
# Load the saved splits from training
test_df = pd.read_csv(os.path.join(config.DATA_DIR, 'test_split.csv'))
print(f'Test set: {len(test_df):,} images')
print(f'Class distribution:')
print(test_df['level'].value_counts().sort_index())

In [ ]:
# Build DataLoader
valid_transforms = get_valid_transforms()
test_dataset = DRDataset(
    test_df,
    images_dir=config.IMAGES_DIR,
    transform=valid_transforms,
    preprocessing=True,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=config.PHASE1_BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=config.PIN_MEMORY,
)

print(f'Test batches: {len(test_loader)}')

In [ ]:
# Load trained model
model = build_model(pretrained=False)

ckpt_path = os.path.join(config.CHECKPOINTS_DIR, 'best_model_phase2.pth')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

print(f'Loaded checkpoint: {ckpt_path}')
print(f'  Epoch: {ckpt["epoch"] + 1}')
print(f'  Val QWK: {ckpt["metrics"]["kappa"]:.4f}')
print(f'  Val Acc: {ckpt["metrics"]["acc"]:.4f}')

## 2. Full Evaluation

In [ ]:
metrics = full_evaluation(
    model=model,
    test_loader=test_loader,
    device=device,
    test_dataset=test_dataset,
    save_dir=config.FIGURES_DIR,
)

## 3. Training Curves

In [ ]:
import json

history_path = os.path.join(config.LOGS_DIR, 'training_history.json')
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    plot_training_curves(history, save_path=os.path.join(config.FIGURES_DIR, 'training_curves.png'))
else:
    print('No training history found. Run notebook 03 first.')

## 4. Single-Image Inference

In [ ]:
from src.evaluate import predict_single_image

def find_image(images_dir, image_id):
    """Try common extensions and return the file path, or None."""
    for ext in ('.jpeg', '.jpg', '.png', '.JPEG', '.JPG'):
        p = os.path.join(images_dir, image_id + ext)
        if os.path.isfile(p):
            return p
    return None

# Pick a random test image
sample_row = test_df.sample(1, random_state=config.SEED).iloc[0]
sample_path = find_image(config.IMAGES_DIR, sample_row['image'])

if sample_path is None:
    print(f"ERROR: Could not find image for '{sample_row['image']}' in {config.IMAGES_DIR}")
else:
    print(f"Image: {sample_row['image']}  |  True label: {config.CLASS_NAMES[sample_row['level']]}")
    result = predict_single_image(
        model, sample_path, device,
        save_path=os.path.join(config.FIGURES_DIR, 'single_prediction.png'),
    )
    print(f"\n  {result['severity']}")
    print(f"  Confidence: {result['confidence']:.1%}")

## 5. Test-Time Augmentation (TTA)

TTA averages predictions across multiple geometric transforms of the same image, typically boosting accuracy by 0.5-1%.

In [ ]:
from src.evaluate import predict_with_tta, compute_metrics, predict

# Standard prediction (no TTA)
y_true, y_pred_std, y_proba_std = predict(model, test_loader, device)
metrics_std = compute_metrics(y_true, y_pred_std, y_proba_std)

# TTA prediction
y_true_tta, y_pred_tta, y_proba_tta = predict_with_tta(
    model, test_df, config.IMAGES_DIR, device,
)
metrics_tta = compute_metrics(y_true_tta, y_pred_tta, y_proba_tta)

# Comparison table
import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'QWK', 'ROC-AUC'],
    'Standard': [
        f"{metrics_std['accuracy']:.4f}",
        f"{metrics_std['qwk']:.4f}",
        f"{metrics_std['roc_auc_weighted']:.4f}",
    ],
    'With TTA': [
        f"{metrics_tta['accuracy']:.4f}",
        f"{metrics_tta['qwk']:.4f}",
        f"{metrics_tta['roc_auc_weighted']:.4f}",
    ],
})
print(comparison.to_string(index=False))

## 6. Summary

Key metrics to report:
- **QWK** (Quadratic Weighted Kappa): the gold standard for DR grading
- **Per-class F1**: shows how well the model handles each severity level
- **ROC-AUC**: overall discriminative ability
- TTA provides a free accuracy boost at inference time
- The confusion matrix reveals common misclassification patterns (typically adjacent grades)